# Session 4: Day 1 Lab Review & Integration (Student Notebook)

This capstone session integrates all Day 1 concepts — LLM foundations (Session 1), prompt engineering (Session 2), and model evaluation (Session 3) — into practical consulting AI exercises.

## Learning Objectives

By the end of this session, you will be able to:

1. Combine prompting, API usage, and evaluation into end-to-end consulting AI workflows
2. Design a multi-component consulting AI system using Day 1 techniques
3. Evaluate AI-generated consulting deliverables against McKinsey quality standards
4. Estimate costs and make model selection decisions for consulting use cases
5. Preview how Day 2 frameworks (LangChain, LangGraph) build on these foundations

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

import openai
import json
import os
import time
import numpy as np

client = openai.OpenAI()
print(f"Model: {os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')}")
print("Setup complete.")

---
## Demo 1: End-to-End Consulting Analysis Pipeline

This demo combines API calls (Session 1), structured prompting (Session 2), and automated evaluation (Session 3) into a single pipeline that generates and quality-checks a consulting deliverable.

In [ ]:
# Demo 1: End-to-End Consulting Analysis Pipeline
# Combines: API calls (S1) + Structured prompting (S2) + G-Eval scoring (S3)

def g_eval_quick(question, response, criterion, description):
    """Lightweight G-Eval scorer (from Session 3)."""
    prompt = f"""Rate this consulting response on {criterion} (1-5).
Criterion: {description}
Question: {question}
Response: {response}
Return JSON with 'score' (1-5) and 'reasoning' (one sentence)."""
    result = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        temperature=0, max_tokens=200
    )
    return json.loads(result.choices[0].message.content)


def consulting_pipeline(question, persona='McKinsey senior consultant'):
    """Generate a consulting analysis, then auto-evaluate it."""
    print(f"Question: {question}")
    print(f"Persona: {persona}")
    print("-" * 50)
    
    # Step 1: Generate (Session 1 + Session 2 techniques)
    start = time.time()
    response = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': f'You are a {persona}. Provide structured, MECE, data-driven analysis with specific recommendations.'},
            {'role': 'user', 'content': question}
        ],
        temperature=0, max_tokens=500
    )
    latency = round((time.time() - start) * 1000)
    content = response.choices[0].message.content
    tokens = response.usage.total_tokens
    
    print(f"\nGenerated Response ({latency}ms, {tokens} tokens):")
    print(content[:300] + '...' if len(content) > 300 else content)
    
    # Step 2: Evaluate (Session 3 techniques)
    criteria = {
        'MECE Structure': 'Response breaks down the problem into mutually exclusive, collectively exhaustive categories.',
        'Actionability': 'Recommendations are specific, prioritized, and implementable.',
        'Executive Readiness': 'Output is polished enough for a C-suite presentation.'
    }
    
    print(f"\nQuality Evaluation:")
    total = 0
    for crit_name, crit_desc in criteria.items():
        result = g_eval_quick(question, content, crit_name, crit_desc)
        score = result.get('score', 0)
        total += score
        bar = '\u2588' * score + '\u2591' * (5 - score)
        print(f"  {crit_name:20s} {bar} {score}/5")
    
    avg = round(total / len(criteria), 1)
    verdict = 'PARTNER-READY' if avg >= 4.0 else 'NEEDS REVISION' if avg >= 3.0 else 'REWORK'
    print(f"\n  Overall: {avg}/5.0 | Verdict: {verdict}")
    return {'content': content, 'avg_score': avg, 'verdict': verdict, 'latency_ms': latency, 'tokens': tokens}


# Run the pipeline
result = consulting_pipeline(
    'What are the key value creation levers for a PE-backed healthcare services platform?'
)


---
## Demo 2: Cost-Quality Trade-off Analysis

For production consulting AI, we need to balance quality against cost and latency. This demo runs the same query at different configurations and plots the trade-off to find the optimal operating point.

In [ ]:
# Demo 2: Cost-Quality Trade-off Analysis
# Combines: Cost estimation (S1) + Temperature comparison (S1) + Evaluation (S3)

import pandas as pd

PRICING = {
    'gpt-4o-mini': {'input': 0.15 / 1_000_000, 'output': 0.60 / 1_000_000},
    'gpt-4o': {'input': 2.50 / 1_000_000, 'output': 10.00 / 1_000_000},
}

question = 'How should a mid-size European bank approach digital transformation to improve customer acquisition?'
configs = [
    {'temperature': 0.0, 'max_tokens': 200, 'label': 'Fast/Cheap (T=0, 200 tok)'},
    {'temperature': 0.0, 'max_tokens': 500, 'label': 'Standard (T=0, 500 tok)'},
    {'temperature': 0.7, 'max_tokens': 500, 'label': 'Creative (T=0.7, 500 tok)'},
]

results = []
for config in configs:
    start = time.time()
    resp = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': 'You are a McKinsey digital strategy expert.'},
            {'role': 'user', 'content': question}
        ],
        temperature=config['temperature'],
        max_tokens=config['max_tokens']
    )
    latency = round((time.time() - start) * 1000)
    content = resp.choices[0].message.content
    usage = resp.usage
    
    # Score
    score = g_eval_quick(question, content, 'Overall Quality',
        'Response is structured, actionable, and ready for executive review.')['score']
    
    # Cost
    model_name = os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')
    cost = 0
    if model_name in PRICING:
        cost = usage.prompt_tokens * PRICING[model_name]['input'] + usage.completion_tokens * PRICING[model_name]['output']
    
    results.append({
        'Config': config['label'],
        'Quality': f"{score}/5",
        'Latency': f"{latency}ms",
        'Tokens': usage.total_tokens,
        'Cost': f"${cost:.6f}",
        'Response Length': len(content),
    })

df = pd.DataFrame(results)
print('COST-QUALITY TRADE-OFF ANALYSIS')
print('=' * 70)
print(df.to_string(index=False))
print('\nObservation: Higher max_tokens increases quality but also cost.')
print('Temperature 0.7 may add creativity but reduces consistency.')


---
## Demo 3: Day 2 Preview — From Raw API Calls to LangChain

Today we used raw OpenAI API calls. On Day 2, we will use **LangChain** to compose reusable pipelines. This demo shows the same task done both ways, previewing the abstraction power of chains.

In [ ]:
# Demo 3: Day 2 Preview — Raw API vs LangChain Pattern

# --- Approach A: Raw API (what we did today) ---
print('APPROACH A: Raw OpenAI API (Day 1 style)')
print('=' * 50)

system_msg = 'You are a McKinsey strategy consultant. Be concise and structured.'
user_msg = 'List 3 key risks in a cross-border pharmaceutical acquisition.'

response = client.chat.completions.create(
    model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
    messages=[
        {'role': 'system', 'content': system_msg},
        {'role': 'user', 'content': user_msg}
    ],
    temperature=0, max_tokens=300
)
print(response.choices[0].message.content)

# --- Approach B: LangChain pattern (Day 2 preview) ---
print(f'\n{"=" * 50}')
print('APPROACH B: LangChain LCEL Chain (Day 2 style)')
print('=' * 50)

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    # Same task, but composable and reusable
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a McKinsey {practice} consultant. Be concise and structured.'),
        ('user', '{question}')
    ])

    llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'), temperature=0)
    chain = prompt | llm | StrOutputParser()

    # Reusable across different practices and questions
    result = chain.invoke({'practice': 'strategy', 'question': user_msg})
    print(result)
    
    print('\nKey advantage: The chain is REUSABLE with different variables.')
    result2 = chain.invoke({'practice': 'operations', 'question': 'What are 3 supply chain optimization levers for a CPG company?'})
    print(f'\nOperations example: {result2[:150]}...')

except ImportError:
    print('LangChain not installed. Run: pip install langchain langchain-openai')
    print('You will set this up fully on Day 2.')

print('\n' + '=' * 50)
print('Day 2 will cover: LCEL chains, custom tools, StateGraph workflows, multi-agent systems')


---
## Task 1: Build an End-to-End Consulting Deliverable Generator

Combine everything from Day 1 into a function that takes a client question, generates a structured consulting response using role-based prompting, auto-evaluates quality, and returns a full deliverable package with metadata.

| Difficulty | Intermediate |
|---|---|


In [ ]:
# Task 1 - Build an End-to-End Consulting Deliverable Generator
#
# TODO: Build a function generate_deliverable(question, practice, review=True) that:
#   1. Selects a persona based on practice area (Strategy, Operations, Digital, M&A)
#   2. Generates a structured response using the persona as system prompt
#   3. Estimates cost using PRICING dict from Demo 2
#   4. If review=True, auto-evaluates with g_eval_quick on 3 criteria
#   5. Returns a dict with content, scores, cost, verdict
#
# Hint: Define a personas dict mapping practice names to system prompts
# Hint: Reuse g_eval_quick() from Demo 1 for evaluation
# Hint: Verdict: avg >= 4.0 PARTNER-READY, >= 3.0 NEEDS REVISION, else REWORK
# Hint: Track latency with time.time() and tokens from response.usage

def generate_deliverable(question, practice='Strategy', review=True):
    """Generate a consulting deliverable with auto-evaluation."""
    # YOUR CODE HERE
    pass

# Test:
# d1 = generate_deliverable('How should a luxury retailer enter the Middle East market?', 'Strategy')
# d2 = generate_deliverable('What are the Day-1 integration priorities for a healthcare acquisition?', 'M&A')


---
## Task 2: Build a Multi-Persona Comparison for a Client Question

A real McKinsey engagement involves multiple expert perspectives. Build a function that generates responses from different practice-area personas for the same question, evaluates each, and recommends which perspective is strongest.

| Difficulty | Intermediate |
|---|---|


In [ ]:
# Task 2 - Build a Multi-Persona Comparison
#
# TODO: Build a function compare_perspectives(question, practices) that:
#   1. Runs generate_deliverable() for each practice area
#   2. Collects avg_score, verdict, tokens, cost for each
#   3. Ranks results by quality score (descending)
#   4. Recommends the strongest perspective
#
# Hint: Default practices = ['Strategy', 'Operations', 'Digital', 'M&A']
# Hint: Use pd.DataFrame(results).sort_values('Avg Score', ascending=False)
# Hint: Best = df.iloc[0] (first row after sorting)

def compare_perspectives(question, practices=None):
    """Compare responses from different McKinsey practice perspectives."""
    # YOUR CODE HERE
    pass

# Test:
# comparison = compare_perspectives(
#     'How should a regional hospital network respond to declining inpatient volumes?'
# )


---
## Task 3: Build a Consulting AI Readiness Assessment

Before deploying an AI system for consulting work, you need to assess its readiness across multiple dimensions. Build a function that runs a test suite of consulting questions and produces a readiness scorecard.

| Difficulty | Advanced |
|---|---|


In [ ]:
# Task 3 - Build a Consulting AI Readiness Assessment
#
# TODO: Build a function readiness_assessment(model_config) that:
#   1. Defines a test suite of 4+ consulting questions across categories
#      (Market Sizing, Due Diligence, Cost Optimization, Strategy)
#   2. Generates a response for each question
#   3. Scores each with g_eval_quick on Quality and Actionability
#   4. Marks each category as PASS (avg >= 3.5) or FAIL
#   5. Prints a readiness scorecard with overall verdict
#
# Hint: Store results as {category: {'score': avg, 'status': 'PASS'/'FAIL'}}
# Hint: Use np.mean() for overall score
# Hint: Verdict: all pass + avg >= 4.0 = READY, >= 75% pass = CONDITIONAL, else NOT READY

def readiness_assessment(model_config=None):
    """Run a comprehensive readiness assessment for consulting AI deployment."""
    # YOUR CODE HERE
    pass

# Test:
# scores = readiness_assessment()


---
## Task 4: Design a Consulting AI System Architecture (On Paper)

This is a design exercise that previews Day 2. Using everything you learned today, design an AI system for a McKinsey use case. Document your design as a structured specification.

| Difficulty | Advanced |
|---|---|


In [ ]:
# Task 4 - Design a Consulting AI System Architecture
#
# TODO: Create a system design dict that maps Day 1 concepts to a practical architecture:
#   1. Define 4 components: Input Router, Analysis Generator, Quality Gate, Knowledge Base
#   2. For each component, specify:
#      - purpose: what it does
#      - technique: which Day 1 technique it uses (API calls, prompting, evaluation, embeddings)
#      - day2_upgrade: how Day 2 frameworks would improve it
#   3. Define model selection (which model/temperature for each component)
#   4. Define quality thresholds (partner-ready, needs revision, rework)
#
# Hint: Router uses classification (T=0, low tokens)
# Hint: Generator uses role-based personas + CoT (T=0, high tokens)
# Hint: Quality Gate uses G-Eval scoring (T=0, medium tokens)
# Hint: Knowledge Base uses embeddings + cosine similarity

system_design = {
    'name': 'McKinsey Engagement AI Assistant',
    'use_case': '# YOUR DESCRIPTION HERE',
    'components': {
        # YOUR CODE HERE — define 4 components
    },
    'model_selection': {
        # YOUR CODE HERE — which model/temp for each task
    },
    'quality_thresholds': {
        # YOUR CODE HERE — scoring thresholds
    },
}

# Print your design
# print(json.dumps(system_design, indent=2))


---
## Summary

In this capstone session, we integrated all Day 1 concepts:

1. **End-to-End Pipeline** — Combined API calls (Session 1), structured prompting (Session 2), and G-Eval scoring (Session 3) into a single consulting deliverable generator
2. **Cost-Quality Trade-offs** — Analyzed how temperature, max_tokens, and model choice affect quality, latency, and cost
3. **Day 2 Preview** — Compared raw API calls with LangChain LCEL chains to preview the composability advantages

### Key Takeaways
- Real consulting AI systems combine multiple techniques from Sessions 1-3
- Auto-evaluation with G-Eval enables quality gates without human review
- Different practice areas benefit from different personas and configurations
- Cost estimation is critical for scoping AI-augmented engagements
- Day 2 will replace raw API calls with LangChain chains, LangGraph workflows, and multi-agent systems

### What's Next (Day 2)
- **Session 1:** Structured Outputs — JSON mode, function calling, Pydantic
- **Session 2:** LangChain — LCEL chains, custom consulting tools
- **Session 3:** LangGraph — StateGraph, conditional edges, cycles
- **Session 4:** Multi-Agent — supervisor-worker teams modeling McKinsey engagement structures